# Task 3 - Cities and points of interest

This task focuses on geographical data processing using Python dictionaries, loops and functions. The solution calculates distances between cities and points of interest, computes city boundaries, assigns POIs to cities and performs simple analysis on customer ratings.

### Short reasoning for our approach
We used dictionaries because they make it easier to connect city names and POI names with coordinates and ratings. We also split the solution into functions so each part of the task is easier to read and test. The distance method stays simple by converting latitude and longitude differences to km and then using Pythagoras. This keeps the solution simple and uses dictionaries, loops and functions from the course.

In [72]:
import math

# converting latitude and longitude to km
LAT_KM = 111
LON_KM = 68

# Norway input range based on https://latitude.to/map/no/norway
NORWAY_LAT_MIN = 57.0
NORWAY_LAT_MAX = 72.0
NORWAY_LON_MIN = 4.0
NORWAY_LON_MAX = 32.0

city_data = {
    'Bergen': [60.364098, 5.404422, 15],
    'Oslo': [59.926959, 10.779454, 10],
    'Trondheim': [63.433654, 10.390277, 5],
    'Lillestrom': [59.9564, 11.0492, 6],
    'Stavanger': [58.9700, 5.7331, 12],
    'Kristiansand': [58.1467, 7.9956, 10]
}

poi_data = {
    (63.426806, 10.396712): ['Nidarosdomen', 4.6],
    (59.926699, 10.701075): ['Vigeland Park', 5.0],
    (59.907704, 10.753163): ['Oslo Opera House', 4.7],
    (60.397561, 5.322833): ['Bryggen', 4.8],
    (60.387639, 5.321623): ['University Museum Bergen', 4.6],
    (60.342901, 5.336796): ['Gamlehaugen', 4.4],
    (59.9573, 11.0480): ['Akershus Art Center', 4.2],
    (59.9560, 11.0508): ['Lillestrom Church', 4.3],
    (59.9546, 11.0501): ['Arasen Stadium', 4.4],
    (58.9722, 5.7314): ['Norwegian Petroleum Museum', 4.7],
    (58.9698, 5.7342): ['Stavanger Cathedral', 4.5],
    (58.9735, 5.7267): ['Old Stavanger', 4.8],
    (58.1468, 7.9959): ['Kristiansand Cathedral', 4.5],
    (58.1455, 8.0055): ['Fiskebrygga', 4.6],
    (58.1463, 8.0127): ['Kilden Performing Arts Centre', 4.8]
}

def distance_km(lat1, lon1, lat2, lon2):
    # convert latitude and longitude differences into km
    # then use Pythagoras to estimate distance
    lat_diff_km = (lat1 - lat2) * LAT_KM
    lon_diff_km = (lon1 - lon2) * LON_KM
    return math.sqrt(lat_diff_km ** 2 + lon_diff_km ** 2)

def is_inside_norway(lat, lon):
    return NORWAY_LAT_MIN <= lat <= NORWAY_LAT_MAX and NORWAY_LON_MIN <= lon <= NORWAY_LON_MAX

def get_norway_input():
    # user gets 3 tries
    for attempt in range(1, 4):
        user_lat = float(input('Enter latitude (57 to 72): '))
        user_lon = float(input('Enter longitude (4 to 32): '))

        if is_inside_norway(user_lat, user_lon):
            return user_lat, user_lon

        if attempt < 3:
            print('Please enter coordinates inside Norway (latitude 57-72, longitude 4-32).')

    print('We ended the program because the user did not enter valid coordinates after 3 tries.')
    raise SystemExit('Program ended.')

def nearest_city(user_lat, user_lon, cities):
    nearest_name = None
    # starts high so first distance becomes the shortest
    min_distance = float('inf')
    for city_name, (city_lat, city_lon, city_size) in cities.items():
        current_distance = distance_km(user_lat, user_lon, city_lat, city_lon)
        if current_distance < min_distance:
            min_distance = current_distance
            nearest_name = city_name
    return nearest_name, round(min_distance, 2)

def nearest_poi(user_lat, user_lon, pois):
    nearest_name = None
    min_distance = float('inf')
    for (poi_lat, poi_lon), (poi_name, poi_rating) in pois.items():
        current_distance = distance_km(user_lat, user_lon, poi_lat, poi_lon)
        if current_distance < min_distance:
            min_distance = current_distance
            nearest_name = poi_name
    return nearest_name, round(min_distance, 2)

def build_city_boundaries(cities):
    city_boundaries = {}
    for city_name, (center_lat, center_lon, city_size_km) in cities.items():
        lat_offset = city_size_km / LAT_KM
        lon_offset = city_size_km / LON_KM
        city_boundaries[city_name] = {
            'center_lat': center_lat,
            'center_lon': center_lon,
            'north': center_lat + lat_offset,
            'south': center_lat - lat_offset,
            'east': center_lon + lon_offset,
            'west': center_lon - lon_offset
        }
    return city_boundaries

def assign_pois(city_boundaries, pois):
    pois_in_city = {}
    for city_name in city_boundaries:
        pois_in_city[city_name] = []
    for (poi_lat, poi_lon), (poi_name, poi_rating) in pois.items():
        for city_name, limits in city_boundaries.items():
            lat_is_inside = limits['south'] <= poi_lat <= limits['north']
            lon_is_inside = limits['west'] <= poi_lon <= limits['east']
            # save poi to city if coordinates are inside city area
            if lat_is_inside and lon_is_inside:
                pois_in_city[city_name].append((poi_name, poi_rating, poi_lat, poi_lon))
    return pois_in_city

def average_rating_by_city(pois_in_city):
    city_average = {}
    for city_name, poi_list in pois_in_city.items():
        # no pois found in city
        if len(poi_list) == 0:
            city_average[city_name] = None
        else:
            total_rating = 0
            for item in poi_list:
                total_rating = total_rating + item[1]
            city_average[city_name] = round(total_rating / len(poi_list), 2)
    return city_average

def sort_pois_by_distance(city_boundaries, pois_in_city):
    ordered_pois = {}
    for city_name, poi_list in pois_in_city.items():
        center_lat = city_boundaries[city_name]['center_lat']
        center_lon = city_boundaries[city_name]['center_lon']
        # sort pois by distance from city center
        sorted_list = sorted(
            poi_list,
            key=lambda item: distance_km(center_lat, center_lon, item[2], item[3])
        )
        ordered_pois[city_name] = []
        for item in sorted_list:
            ordered_pois[city_name].append(item[0])
    return ordered_pois

## a) Find nearest city and POI

The first part of the task calculates the nearest city and point of interest based on user coordinates. Latitude and longitude differences are converted into kilometers using the approximations provided in the assignment. The Pythagorean theorem is then used to estimate the distance between coordinates.

In [78]:
input_lat, input_lon = get_norway_input()
print(f'Chosen input: latitude={input_lat}, longitude={input_lon}')

closest_city_name, closest_city_distance = nearest_city(input_lat, input_lon, city_data)
closest_poi_name, closest_poi_distance = nearest_poi(input_lat, input_lon, poi_data)

print(f'Closest city: {closest_city_name} ({closest_city_distance} km)')
print(f'Closest POI: {closest_poi_name} ({closest_poi_distance} km)')

Chosen input: latitude=57.0, longitude=5.0
Closest city: Stavanger (224.28 km)
Closest POI: Stavanger Cathedral (224.28 km)


The program validates user input to make sure the coordinates are inside Norway. The number of attempts is limited to avoid an infinite loop caused by invalid input.

## b) Compute city boundaries

The city boundaries are calculated by converting the city size from kilometers into latitude and longitude offsets. The task assumes square city boundaries and ignores the curvature of the Earth.

In [74]:
city_boundaries = build_city_boundaries(city_data)

print('City boundaries:')
for city_name, limits in city_boundaries.items():
    print(city_name, {
        'north': round(limits['north'], 6),
        'south': round(limits['south'], 6),
        'east': round(limits['east'], 6),
        'west': round(limits['west'], 6)
    })

City boundaries:
Bergen {'north': 60.499233, 'south': 60.228963, 'east': 5.62501, 'west': 5.183834}
Oslo {'north': 60.017049, 'south': 59.836869, 'east': 10.926513, 'west': 10.632395}
Trondheim {'north': 63.478699, 'south': 63.388609, 'east': 10.463806, 'west': 10.316748}
Lillestrom {'north': 60.010454, 'south': 59.902346, 'east': 11.137435, 'west': 10.960965}
Stavanger {'north': 59.078108, 'south': 58.861892, 'east': 5.909571, 'west': 5.556629}
Kristiansand {'north': 58.23679, 'south': 58.05661, 'east': 8.142659, 'west': 7.848541}


Using square boundaries simplifies the implementation and makes it easier to check whether coordinates belong to a city. In reality, city borders are usually irregular, but the square approach matches the assumptions in the assignment.

## c) Assign POIs to cities

This part checks whether each POI is located inside the calculated city boundaries. A POI belongs to a city if its latitude and longitude are inside the north, south, east and west limits.

In [75]:
pois_in_city = assign_pois(city_boundaries, poi_data)

print('POIs in each city:')
for city_name, poi_list in pois_in_city.items():
    poi_names = []
    for item in poi_list:
        poi_names.append(item[0])
    print(f'{city_name}: {poi_names}')

POIs in each city:
Bergen: ['Bryggen', 'University Museum Bergen', 'Gamlehaugen']
Oslo: ['Vigeland Park', 'Oslo Opera House']
Trondheim: ['Nidarosdomen']
Lillestrom: ['Akershus Art Center', 'Lillestrom Church', 'Arasen Stadium']
Stavanger: ['Norwegian Petroleum Museum', 'Stavanger Cathedral', 'Old Stavanger']
Kristiansand: ['Kristiansand Cathedral', 'Fiskebrygga', 'Kilden Performing Arts Centre']


Dictionaries were used because they provide direct access to cities and coordinates through keys. This made the data easier to organize and retrieve during the calculations.

## d) Compute average POI rating per city

The average rating for each city is calculated by summing all POI ratings inside the city and dividing by the number of POIs.

In [76]:
city_average_rating = average_rating_by_city(pois_in_city)
print('Average rating per city:')
for city_name, rating in city_average_rating.items():
    print(f'Average rating in {city_name}: {rating}')

Average rating per city:
Average rating in Bergen: 4.6
Average rating in Oslo: 4.85
Average rating in Trondheim: 4.6
Average rating in Lillestrom: 4.3
Average rating in Stavanger: 4.67
Average rating in Kristiansand: 4.63


The solution also handles cities without POIs by returning None as the average rating.

## e) Order POIs by distance from city center

The POIs are sorted based on their distance from the city center, from nearest to farthest.

In [77]:
ordered_pois = sort_pois_by_distance(city_boundaries, pois_in_city)
print('POIs ordered by distance from city center:')
for city_name, ordered_list in ordered_pois.items():
    print(f'{city_name}: {ordered_list}')

POIs ordered by distance from city center:
Bergen: ['Gamlehaugen', 'University Museum Bergen', 'Bryggen']
Oslo: ['Oslo Opera House', 'Vigeland Park']
Trondheim: ['Nidarosdomen']
Lillestrom: ['Lillestrom Church', 'Akershus Art Center', 'Arasen Stadium']
Stavanger: ['Stavanger Cathedral', 'Norwegian Petroleum Museum', 'Old Stavanger']
Kristiansand: ['Kristiansand Cathedral', 'Fiskebrygga', 'Kilden Performing Arts Centre']


### Reflection: choices and limitations

The solution uses dictionaries because they make it easier to organize cities, coordinates and POI information. Functions were also used to split the task into smaller and more readable parts, which made the workflow easier to follow and reduced repeated code.

The distance calculation is simplified by using fixed km per degree values together with the Pythagorean theorem. This works well for the assignment, but it is not fully accurate for real geographical calculations because the curvature of the Earth is ignored. The city boundaries are also simplified into square areas around each city center.

The program includes some basic edge-case handling. Invalid coordinate input is limited to three attempts, and cities without POIs return None as the average rating. Overall, the implementation focuses on readability and simplicity while still solving all required parts of the assignment.

## Conclusion

The solution uses Python dictionaries, loops and functions to solve different geographical tasks related to cities and points of interest. Dictionaries made it easier to organize city names, coordinates and ratings, while functions helped split the problem into smaller and more readable parts.

The program calculates approximate distances by converting latitude and longitude differences into kilometers and then using the Pythagorean theorem. This keeps the implementation simple and follows the assumptions given in the assignment. However, the calculations are not perfectly accurate for real geographical distances because the curvature of the Earth is ignored.

The implementation demonstrates how boundaries can be used to determine whether a POI belongs to a city. Using square city boundaries simplified the calculations, even though real cities usually have more irregular shapes.

Overall, the task shows how basic Python programming concepts such as dictionaries, lists, loops, conditions and functions can be combined to perform simple data manipulation and geographical analysis.